In [1]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus']=False

import pandas as pd
import numpy as np
from sklearn import svm,metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression 
from sklearn.preprocessing import PolynomialFeatures  # 다항특성 자동생성
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error   # 평균절대값오차 오차범위
from sklearn.preprocessing import StandardScaler  # 정규화클래스
from sklearn.model_selection import train_test_split

In [2]:
# 1. 데이터 불러오기 및 X, y 분리
df = pd.read_csv('data/MultipleLinearRegressionData.csv')
data = df.iloc[:, :-2].values  # 수치형 데이터
label = df.iloc[:, -1].values  # 정답 (score)

In [3]:
# 2. 범주형 데이터(place) 원-핫 인코딩
place = df['place'].to_numpy().reshape(-1, 1)
encoder = OneHotEncoder(sparse_output=False, drop='first')
result = encoder.fit_transform(place)

In [4]:
# 데이터 합치기 (수치형 + 원핫인코딩된 장소)
total_data = np.concatenate((data, result), axis=1)

In [5]:
# 3. 훈련 / 테스트 데이터 분리 (random_state로 고정)
train_data, test_data, train_label, test_label = train_test_split(
    total_data, label, random_state=42, test_size=0.2
)

In [6]:
# 4. 다항 특성 자동 생성 (degree=5)
poly = PolynomialFeatures(degree=5, include_bias=False)
poly.fit(train_data)  # 훈련 데이터로 기준(설계도) 생성
train_poly = poly.transform(train_data)
test_poly = poly.transform(test_data)   # 동일한 기준 적용

In [7]:
# 5. 데이터 표준화 (스케일링)
ss = StandardScaler()
ss.fit(train_poly)    # 훈련 데이터 기준으로 평균/분산 계산
train_scaled = ss.transform(train_poly)
test_scaled = ss.transform(test_poly)

In [8]:
# 6. 릿지(Ridge) 모델로 최적의 alpha 찾기 및 학습
alpha_list = [0.001, 0.01, 0.1, 1, 10, 100]
for alpha in alpha_list:
    ridge = Ridge(alpha=alpha)
    ridge.fit(train_scaled, train_label)
    print(f"alpha: {alpha}")
    print(f"훈련 점수: {ridge.score(train_scaled, train_label):.4f}")
    print(f"테스트 점수: {ridge.score(test_scaled, test_label):.4f}")

alpha: 0.001
훈련 점수: 1.0000
테스트 점수: -1.4201
alpha: 0.01
훈련 점수: 0.9998
테스트 점수: -1.9876
alpha: 0.1
훈련 점수: 0.9988
테스트 점수: -0.2876
alpha: 1
훈련 점수: 0.9864
테스트 점수: 0.9339
alpha: 10
훈련 점수: 0.8877
테스트 점수: 0.8607
alpha: 100
훈련 점수: 0.6908
테스트 점수: -0.1593
